# Notebook 04 – Build Gold Analytical Model

## Objective
Organize cleansed Silver operational data into a star-schema Gold analytical model optimized for Power BI reporting and business analytics.

# Business Model

Dimensions
- Customers
- Products
- Calendar
- Regions
- Category

Fact Tables
- Sales
- CampaignPerformance
- SalesTargets

## Pipeline Position
This is **Notebook 04** in the Medallion pipeline. It consumes cleansed `silver_*` Delta tables produced by Notebook 03 and reshapes them into Gold layer dimension and fact tables. These Gold Delta tables serve as the Direct Lake source for the Fabric semantic model and Power BI reporting layers.

## Process
1. Generate Dimension tables (`Calendar`, `Customers`, `Products`, `Regions`, `Category`).
2. Generate Fact tables (`Sales`, `marketing_campaign`, `SalesTarget`).
3. Validate data types and persist Gold Delta tables.

# Section 1 - Analytical Dimensions Creation

## Objective
Build star-schema dimension tables (`Calendar`, `Customers`, `Products`, `Regions`, `Category`) from cleansed Silver entities to provide standardized filtering and grouping attributes for reporting.

## Calendar

Create a reusable calendar dimension from the Sales transaction dates.

In [1]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


sales = spark.table("silver_orders")
campaigns = spark.table("silver_campaign_performance")


# Get dates from sales
sales_dates = sales.select(
    col("OrderDate").cast("date").alias("Date")
)


# Get dates from campaigns
campaign_dates = campaigns.select(
    col("CampaignDate").cast("date").alias("Date")
)


# Combine all dates
all_dates = sales_dates.union(campaign_dates)


# Get minimum and maximum dates
date_range = all_dates.agg(
    min("Date").alias("MinDate"),
    max("Date").alias("MaxDate")
).collect()[0]


# Create calendar table
calendar = (
    spark.sql(
        f"""
        SELECT explode(
            sequence(
                to_date('{date_range["MinDate"]}'),
                to_date('{date_range["MaxDate"]}'),
                interval 1 day
            )
        ) AS Date
        """
    )
    .withColumn("Year", year("Date"))
    .withColumn("Quarter", quarter("Date"))
    .withColumn("MonthNumber", month("Date"))
    .withColumn("MonthName", date_format("Date", "MMM"))
    .withColumn("Day", dayofmonth("Date"))
    .withColumn(
        "YearMonthKey",
        year("Date") * 100 + month("Date")
    )
)


# Write calendar table to Delta
calendar.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("Calendar")


# Display calendar
display(spark.table("Calendar"))

StatementMeta(, 946a75e4-5971-4027-9ce2-cc3249b521b2, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, aa97da83-19fc-430c-b66f-78d7fe031504)

### Customers Dimension
Extracts core customer demographic fields from `silver_customers` and persists the `Customers` Gold dimension table.

In [1]:
customers = spark.table("silver_customers")

customers = customers.select(
    "CustomerID",
    "CustomerName",
    "Email",
    "CustomerType",
    "JoinDate"
)

customers.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("Customers")

display(spark.table("Customers"))

StatementMeta(, 25baae36-db01-4bf2-ad78-7c09b867ed13, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 34dcdadb-48ab-4e94-96e8-55e5968c37a6)

### Products Dimension
Extracts product attributes, categories, and standard pricing from `silver_products` and persists the `Products` Gold dimension table.

In [1]:
products = spark.table("silver_products")

products = products.select(
    "ProductID",
    "ProductName",
    "Category",
    "UnitPrice",
    "StandardCost"
)

products.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("Products")

display(spark.table("Products"))

StatementMeta(, 3370cabb-c38f-4924-b66b-70635e3cd2dd, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2e019cee-65a9-48b3-b529-d1c532b80893)

### Regions Dimension Creation
Generates a distinct list of operating regions from `silver_customers` and assigns a surrogate primary key (`RegionID`) to establish a formal regional dimension table.

In [7]:
from pyspark.sql.functions import *

regions = spark.table("silver_customers")

regions = regions.select("Region").distinct()

regions = regions.withColumn(
    "RegionID",
    monotonically_increasing_id() + 1
)

regions = regions.select(
    "RegionID",
    "Region"
)

regions.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("Regions")

display(spark.table("Regions"))

StatementMeta(, fdb03e8a-ad0c-4628-a75e-b9eeeaa2351f, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 98413324-b06b-4d2d-891c-470428fc4b56)

# Section 2 - Analytical Fact Tables Creation

## Objective
Construct core business process fact tables (`Sales`, `marketing_campaign`, `SalesTarget`) by joining Silver transaction headers and line items, enforcing key relationships, and formatting numeric measures for reporting.

### Sales Fact Table
Joins `silver_orders` and `silver_order_lines` with dimension lookups to construct the primary transactional `Sales` fact table containing order measures, pricing metrics, and foreign keys.

In [2]:
from pyspark.sql.functions import *


orders = spark.table("silver_orders")
order_lines = spark.table("silver_order_lines")
customers = spark.table("silver_customers")
products = spark.table("Products")


sales = (
    order_lines.alias("ol")
    .join(
        orders.alias("o"),
        col("ol.OrderID") == col("o.OrderID"),
        "inner"
    )
    .join(
        customers.alias("c"),
        col("o.CustomerID") == col("c.CustomerID"),
        "inner"
    )
    .join(
        products.alias("p"),
        col("ol.ProductID") == col("p.ProductID"),
        "inner"
    )
    .select(
        col("ol.OrderLineID"),
        col("o.OrderID"),
        col("o.OrderDate"),
        col("o.CustomerID"),
        col("c.Region").alias("Region"),
        col("ol.ProductID"),
        col("ol.Quantity").cast("int").alias("Quantity"),
        col("ol.UnitPrice").cast("decimal(18,2)").alias("UnitPrice"),
        col("ol.CostAtSale").cast("decimal(18,2)").alias("CostAtSale"),
        col("ol.Discount").cast("decimal(18,2)").alias("Discount"),
        col("ol.LineAmount").cast("decimal(18,2)").alias("LineAmount"),
        col("o.Channel")
    )
)


sales.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("Sales")


display(spark.table("Sales"))

StatementMeta(, 272863bb-1f85-460e-b75b-aa192e39506a, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a3ad5325-a2db-42a3-994a-ee147d3ff183)

# Marketing Campaign

## Source

silver_campaign_performance

## Grain

One row per CampaignID.

## Output Columns

- CampaignID
- CampaignName
- CampaignDate
- Platform
- ProductCategory
- Impressions
- Clicks
- Conversions
- Spend
- RevenueGenerated

In [3]:
from pyspark.sql.functions import col, to_date

campaigns = spark.table("silver_campaign_performance")

gold_campaigns = campaigns

gold_campaigns = gold_campaigns.withColumn(
    "CampaignDate",
    to_date(col("CampaignDate"))
)

gold_campaigns = gold_campaigns.withColumn(
    "Clicks",
    col("Clicks").cast("int")
)

gold_campaigns = gold_campaigns.withColumn(
    "Conversions",
    col("Conversions").cast("int")
)

gold_campaigns = gold_campaigns.withColumn(
    "Impressions",
    col("Impressions").cast("int")
)

gold_campaigns = gold_campaigns.withColumn(
    "RevenueGenerated",
    col("RevenueGenerated").cast("decimal(18,2)")
)

gold_campaigns = gold_campaigns.withColumn(
    "Spend",
    col("Spend").cast("decimal(18,2)")
)

gold_campaigns = gold_campaigns.select(
    "CampaignID",
    "CampaignName",
    "CampaignDate",
    "Platform",
    "Region",
    "ProductCategory",
    "Impressions",
    "Clicks",
    "Conversions",
    "Spend",
    "RevenueGenerated"
)

gold_campaigns.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("marketing_campaign")

display(spark.table("marketing_campaign"))

StatementMeta(, a87fb9bb-3759-4747-a07a-c46fc95e0dab, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b77dcd69-4df6-4ab0-bf32-a69bc0105f89)

# Sales Target

## Purpose

Create the Gold-layer sales target table from the cleaned Silver sales target data.

## Source

- `silver_sales_targets`

## Grain

One row per:

**Year + Month + Region + ProductCategory**

## Transformations

- Convert `Year` to integer.
- Convert `Month` to integer.
- Convert `SalesTarget` to decimal.
- Preserve the cleaned `Region` and `ProductCategory`.
- Retain `TargetID` as the row identifier.

## Gold Table

`SalesTarget`

## Output Columns

- TargetID
- Year
- Month
- Region
- ProductCategory
- SalesTarget

## Architectural Note

`SalesTarget` remains a separate analytical table from `Sales`.

Its grain is monthly target by region and product category. It should not contain CustomerID or ProductID because the source target data does not have those keys.

In [1]:
from pyspark.sql.functions import col

targets = spark.table("silver_sales_targets")

gold_targets = (
    targets
    .withColumn("Year", col("Year").cast("int"))
    .withColumn("Month", col("Month").cast("int"))
    .withColumn(
        "YearMonthKey",
        col("Year") * 100 + col("Month")
    )
    .withColumn("SalesTarget", col("SalesTarget").cast("decimal(18,2)"))
    .select(
        "TargetID",
        "Year",
        "Month",
        "YearMonthKey",
        "Region",
        "ProductCategory",
        "SalesTarget"
    )
)

gold_targets.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("SalesTarget")

display(spark.table("SalesTarget"))

StatementMeta(, c62da39a-2bf4-47ff-a37c-1a4919ad7209, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c114ec79-dacf-4a6e-b50d-8c285f20ad58)

### Sales Date Validation & Type Enforcement
Converts `OrderDate` strings into explicit `date` data types and verifies that zero null conversion failures occurred before overwriting the `Sales` Gold table.

In [5]:
from pyspark.sql.functions import col, to_date

sales = spark.table("Sales")

# Convert OrderDate from string to date
sales_converted = sales.withColumn(
    "OrderDate",
    to_date(col("OrderDate"), "yyyy-MM-dd")
)

# Validate conversion before overwrite
invalid_dates = sales_converted.filter(
    col("OrderDate").isNull()
).count()

print(f"Invalid OrderDate conversions: {invalid_dates}")

if invalid_dates == 0:

    sales_converted.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("Sales")

    print("Sales table updated successfully.")

    display(spark.table("Sales").select("OrderDate").limit(10))

else:

    print("Sales table NOT overwritten.")
    print("OrderDate conversion produced null values.")

StatementMeta(, b016eb0e-fba1-4d92-90e4-77e3a01bbe97, 7, Finished, Available, Finished, False)

Invalid OrderDate conversions: 0
Sales table updated successfully.


SynapseWidget(Synapse.DataFrame, 23c86266-a4ea-4c0c-98fe-ec007a20a731)

### Category Dimension Creation
Consolidates distinct product category strings from `products`, `Sales`, and `marketing_campaign` tables into a unified `Category` dimension table.

In [ ]:
from pyspark.sql.functions import col

product_categories = (
    spark.table("products")
    .select(col("Category"))
    .distinct()
)

sales_categories = (
    spark.table("Sales")
    .select(col("Category"))
    .distinct()
)

campaign_categories = (
    spark.table("marketing_campaign")
    .select(col("ProductCategory").alias("Category"))
    .distinct()
)

category = (
    product_categories
    .union(sales_categories)
    .union(campaign_categories)
    .distinct()
    .orderBy("Category")
)

category.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("Category")

display(spark.table("Category"))